# Обучение моделей

## Импорт библиотек

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer, TargetEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    recall_score,
    precision_score,
    f1_score
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import optuna

# Добавляем путь к src
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import DataLoader
from src.preprocess import Preprocessor
from src.features import FeatureEngineer

## Загрузка данных

In [2]:
# Загружаем датасет
PROJECT_ROOT = project_root
DATASET_PATH = PROJECT_ROOT / "datasets" / "Credit Risk Data.csv"
KAGGLE_DS = "alexdister/credit-risk-dataset"

df = DataLoader.load(DATASET_PATH, KAGGLE_DS)

In [3]:
df.head()

,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,...,city_latitude,city_longitude,employment_type,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
0,CUST_00001,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,...,43.6532,-79.3832,Self-employed,36,0.593220,8402.453850,0.735635,14,0.495557,0
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,...,43.6532,-79.3832,Full-time,36,0.104167,1607.802794,0.271646,10,0.585436,3
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,...,51.6214,-3.9436,Full-time,36,0.572917,2760.505633,0.860469,14,0.750732,0
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,...,49.2827,-123.1207,Part-time,12,0.534351,7155.286150,0.643592,15,0.379333,0
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,...,42.8864,-78.8784,Part-time,36,0.643382,15626.153440,0.930628,4,0.228103,0


## Разделение на выборки

In [4]:
# Разделение на признаки и целевую переменную
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (32581, 28), y shape: (32581,)


In [5]:
# Разделение на обучающую и тестовую выборки (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} записей")
print(f"Test:  {X_test.shape[0]} записей")
print(f"Доля дефолтов в train: {y_train.mean():.4f}")
print(f"Доля дефолтов в test:  {y_test.mean():.4f}")

Train: 26064 записей
Test:  6517 записей
Доля дефолтов в train: 0.2182
Доля дефолтов в test:  0.2182


## Обучение моделей

### LogisticRegression

In [6]:
# Определение групп признаков для логистической регрессии

# Сильно скошенные -> логарифм + масштабирование
lr_log_cols = ["person_income", "loan_amnt", "other_debt"]

# Остальные числовые -> масштабирование (включая loan_term_months и past_delinquencies)
lr_num_cols = [
    "person_age", "person_emp_length", "loan_int_rate", "loan_percent_income",
    "cb_person_cred_hist_length", "debt_to_income_ratio", "open_accounts",
    "credit_utilization_ratio", "income_debt_balance",
    "is_high_debt", "is_high_loan_pct", "is_high_rate", "emp_length_missing",
    "loan_term_months", "past_delinquencies"
]

# Порядковый признак loan_grade -> OrdinalEncoder с явными категориями
grade_categories = [["A", "B", "C", "D", "E", "F", "G"]]
grade_encoder = OrdinalEncoder(
    categories=grade_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Бинарные строковые -> OneHotEncoder с drop='if_binary'
lr_onehot_binary_cols = ["cb_person_default_on_file", "gender"]

# Низкокардинальные строковые (2–10 уникальных) -> OneHotEncoder
lr_onehot_cols = [
    "person_home_ownership", "loan_intent", "marital_status",
    "education_level", "employment_type"
]

# Средне- и высококардинальные -> TargetEncoder
lr_target_cols = ["city", "grade_ownership", "default_grade"]

In [7]:
# Сборка ColumnTransformer
lr_preprocessor = ColumnTransformer([
    ("log_num", Pipeline([
        ("log", FunctionTransformer(np.log1p, validate=True)),
        ("scale", StandardScaler())
    ]), lr_log_cols),
    ("num", StandardScaler(), lr_num_cols),
    ("grade_ordinal", grade_encoder, ["loan_grade"]),
    ("onehot_binary", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), lr_onehot_binary_cols),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), lr_onehot_cols),
    ("target", TargetEncoder(target_type="binary", smooth=1.0), lr_target_cols)
])

In [8]:
# Полный пайплайн для логистической регрессии
lr_pipeline = Pipeline([
    ("basic_preprocessing", Preprocessor()),
    ("feature_engineer", FeatureEngineer(include_weak=False)),
    ("custom_preprocessing", lr_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    ))
])

In [9]:
# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_scores = cross_val_score(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print(f"LogisticRegression CV ROC-AUC: {lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})")

LogisticRegression CV ROC-AUC: 0.8834 (+/- 0.0026)


### RandomForestClassifier

In [10]:
# Определение групп признаков для Random Forest

# Числовые признаки (без масштабирования, passthrough)
rf_num_cols = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length",
    "debt_to_income_ratio", "open_accounts", "credit_utilization_ratio",
    "income_debt_balance", "is_high_debt", "is_high_loan_pct", "is_high_rate",
    "emp_length_missing", "loan_term_months", "past_delinquencies"
]

# Порядковый признак loan_grade -> OrdinalEncoder с явными категориями
grade_categories = [["A", "B", "C", "D", "E", "F", "G"]]
grade_encoder = OrdinalEncoder(
    categories=grade_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Бинарные строковые -> OneHotEncoder с drop='if_binary'
rf_onehot_binary_cols = ["cb_person_default_on_file", "gender"]

# Низкокардинальные строковые (2–10 уникальных) -> OneHotEncoder
rf_onehot_cols = [
    "person_home_ownership", "loan_intent", "marital_status",
    "education_level", "employment_type"
]

# Средне- и высококардинальные -> OneHotEncoder
rf_high_card_cols = ["city", "grade_ownership", "default_grade"]

In [11]:
# Сборка ColumnTransformer
rf_preprocessor = ColumnTransformer([
    ("num", "passthrough", rf_num_cols),
    ("grade_ordinal", grade_encoder, ["loan_grade"]),
    ("onehot_binary", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=True), rf_onehot_binary_cols),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True), rf_onehot_cols),
    ("onehot_high", OneHotEncoder(handle_unknown="ignore", sparse_output=True), rf_high_card_cols)
])

In [12]:
# Полный пайплайн для Random Forest
rf_pipeline = Pipeline([
    ("basic_preprocessing", Preprocessor()),
    ("feature_engineer", FeatureEngineer()),
    ("custom_preprocessing", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

In [13]:
# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_scores = cross_val_score(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print(f"Random Forest CV ROC-AUC: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")

Random Forest CV ROC-AUC: 0.9134 (+/- 0.0037)


### XGBClassifier

In [14]:
# Определение групп признаков для XGBoost

# Числовые признаки (без масштабирования, passthrough)
xgb_num_cols = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length",
    "debt_to_income_ratio", "open_accounts", "credit_utilization_ratio",
    "income_debt_balance", "is_high_debt", "is_high_loan_pct", "is_high_rate",
    "emp_length_missing", "loan_term_months", "past_delinquencies"
]

# Порядковый признак loan_grade -> OrdinalEncoder с явными категориями
grade_categories = [["A", "B", "C", "D", "E", "F", "G"]]
grade_encoder = OrdinalEncoder(
    categories=grade_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Бинарные строковые -> OneHotEncoder с drop='if_binary'
xgb_onehot_binary_cols = ["cb_person_default_on_file", "gender"]

# Низкокардинальные строковые (2–10 уникальных) -> OneHotEncoder
xgb_onehot_cols = [
    "person_home_ownership", "loan_intent", "marital_status",
    "education_level", "employment_type"
]

# Средне- и высококардинальные -> OneHotEncoder
xgb_high_card_cols = ["city", "grade_ownership", "default_grade"]

In [15]:
# Сборка ColumnTransformer
xgb_preprocessor = ColumnTransformer([
    ("num", "passthrough", xgb_num_cols),
    ("grade_ordinal", grade_encoder, ["loan_grade"]),
    ("onehot_binary", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=True), xgb_onehot_binary_cols),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True), xgb_onehot_cols),
    ("onehot_high", OneHotEncoder(handle_unknown="ignore", sparse_output=True), xgb_high_card_cols)
])

In [16]:
# Полный пайплайн для XGBoost
xgb_pipeline = Pipeline([
    ("basic_preprocessing", Preprocessor()),
    ("feature_engineer", FeatureEngineer(include_weak=True)),
    ("custom_preprocessing", xgb_preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=3.5,
        random_state=42,
        eval_metric="logloss"
    ))
])

In [17]:
# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_scores = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print(f"XGBoost CV ROC-AUC: {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})")

XGBoost CV ROC-AUC: 0.9405 (+/- 0.0048)


### LGBMClassifier

In [18]:
# Определение групп признаков для LightGBM

# Числовые признаки (без масштабирования, passthrough)
lgb_num_cols = [
    "person_age", "person_income", "person_emp_length", "loan_amnt",
    "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length",
    "debt_to_income_ratio", "open_accounts", "credit_utilization_ratio",
    "income_debt_balance", "is_high_debt", "is_high_loan_pct", "is_high_rate",
    "emp_length_missing", "loan_term_months", "past_delinquencies"
]

# Порядковый признак loan_grade -> OrdinalEncoder с явными категориями
grade_categories = [["A", "B", "C", "D", "E", "F", "G"]]
grade_encoder = OrdinalEncoder(
    categories=grade_categories,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

# Бинарные строковые -> OneHotEncoder с drop='if_binary'
lgb_onehot_binary_cols = ["cb_person_default_on_file", "gender"]

# Низкокардинальные строковые (2–10 уникальных) -> OneHotEncoder
lgb_onehot_cols = [
    "person_home_ownership", "loan_intent", "marital_status",
    "education_level", "employment_type"
]

# Средне- и высококардинальные -> OneHotEncoder
lgb_high_card_cols = ["city", "grade_ownership", "default_grade"]

In [19]:
# Сборка ColumnTransformer
lgb_preprocessor = ColumnTransformer([
    ("num", "passthrough", lgb_num_cols),
    ("grade_ordinal", grade_encoder, ["loan_grade"]),
    ("onehot_binary", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=True), lgb_onehot_binary_cols),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True), lgb_onehot_cols),
    ("onehot_high", OneHotEncoder(handle_unknown="ignore", sparse_output=True), lgb_high_card_cols)
])

In [20]:
# Полный пайплайн для LightGBM
lgb_pipeline = Pipeline([
    ("basic_preprocessing", Preprocessor()),
    ("feature_engineer", FeatureEngineer(include_weak=True)),
    ("custom_preprocessing", lgb_preprocessor),
    ("classifier", LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=3.5,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ))
])

In [21]:
# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_scores = cross_val_score(
    lgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print(f"LightGBM CV ROC-AUC: {lgb_scores.mean():.4f} (+/- {lgb_scores.std():.4f})")

LightGBM CV ROC-AUC: 0.9406 (+/- 0.0050)


### CatBoostClassifier

In [22]:
# Список категориальных признаков
cat_features = [
    "person_home_ownership", "loan_intent", "loan_grade",
    "cb_person_default_on_file", "gender", "marital_status",
    "education_level", "city", "employment_type",
    "grade_ownership", "default_grade"
]

In [23]:
# Пайплайн
catboost_pipeline = Pipeline([
    ("basic_preprocessing", Preprocessor()),
    ("feature_engineer", FeatureEngineer(include_weak=False)),
    ("classifier", CatBoostClassifier(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        auto_class_weights="Balanced",
        verbose=False,
        random_seed=42
    ))
])

In [24]:
# Стратифицированная кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cat_scores = cross_val_score(
    catboost_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    params={"classifier__cat_features": cat_features}
)

print(f"CatBoost CV ROC-AUC: {cat_scores.mean():.4f} (+/- {cat_scores.std():.4f})")

CatBoost CV ROC-AUC: 0.9396 (+/- 0.0048)


### Результаты обучения моделей

In [25]:
# Сбор результатов кросс-валидации
results = {
    "Logistic Regression": lr_scores,
    "Random Forest": rf_scores,
    "XGBoost": xgb_scores,
    "LightGBM": lgb_scores,
    "CatBoost": cat_scores
}

# Создание DataFrame
cv_results_df = pd.DataFrame({
    "Модель": list(results.keys()),
    "Средний ROC-AUC": [scores.mean() for scores in results.values()],
    "Стандартное отклонение": [scores.std() for scores in results.values()]
}).sort_values("Средний ROC-AUC", ascending=False)

# Вывод таблицы
print("Результаты кросс-валидации (5 фолдов):")
display(cv_results_df.style.format({
    "Средний ROC-AUC": "{:.4f}",
    "Стандартное отклонение": "{:.4f}"
}))

Результаты кросс-валидации (5 фолдов):


,Модель,Средний ROC-AUC,Стандартное отклонение
3,LightGBM,0.9406,0.0050
2,XGBoost,0.9405,0.0048
4,CatBoost,0.9396,0.0048
1,Random Forest,0.9134,0.0037
0,Logistic Regression,0.8834,0.0026


**Вывод по результатам кросс-валидации**

На основе кросс-валидации (5 фолдов) лучшие результаты показали три бустинговые модели: **LightGBM**, **XGBoost** и **CatBoost**. Разница между ними составляет менее 0.001, что статистически незначимо, однако LightGBM обучается значительно быстрее CatBoost и сопоставим с XGBoost по скорости, что делает его предпочтительным выбором. Случайный лес и логистическая регрессия заметно уступают бустингу, что подтверждает наличие нелинейных зависимостей в данных. Стандартные отклонения всех моделей малы (0.002–0.004), что говорит о стабильности качества на разных подвыборках. Таким образом, для финального обучения и внедрения рекомендуется выбрать **LightGBM** как модель с наилучшим соотношением качества и времени обучения.

## Настройка гиперпараметров лучшей модели

In [26]:
# Фиксированный параметр балансировки классов
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Функция оптимизации
def objective(trial):
    params = {
        "classifier__n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=100),
        "classifier__max_depth": trial.suggest_int("max_depth", 3, 12),
        "classifier__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "classifier__num_leaves": trial.suggest_int("num_leaves", 16, 256, step=16),
        "classifier__subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "classifier__colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "classifier__min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
        "classifier__reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "classifier__reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "classifier__scale_pos_weight": scale_pos_weight,
        "classifier__random_state": 42,
        "classifier__n_jobs": -1,
        "classifier__verbose": -1
    }

    lgb_pipeline.set_params(**params)

    scores = cross_val_score(
        lgb_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )
    return scores.mean()

In [27]:
# Запуск оптимизации
study = optuna.create_study(direction="maximize", study_name="lgbm_optimization")
study.optimize(objective, n_trials=50, show_progress_bar=True)

[I 2026-08-12 19:56:53,172] A new study created in memory with name: lgbm_optimization


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-12 19:56:59,644] Trial 0 finished with value: 0.937141337816065 and parameters: {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.014259803449581825, 'num_leaves': 208, 'subsample': 0.7305631555677485, 'colsample_bytree': 0.6890294826715524, 'min_child_samples': 21, 'reg_alpha': 7.799306602430786e-05, 'reg_lambda': 8.02379262697903e-08}. Best is trial 0 with value: 0.937141337816065.
[I 2026-08-12 19:57:08,414] Trial 1 finished with value: 0.9428631738136156 and parameters: {'n_estimators': 800, 'max_depth': 7, 'learning_rate': 0.018739281037579773, 'num_leaves': 208, 'subsample': 0.8749809069328617, 'colsample_bytree': 0.8956396964768717, 'min_child_samples': 21, 'reg_alpha': 2.522808525905855, 'reg_lambda': 1.751245461720895e-07}. Best is trial 1 with value: 0.9428631738136156.
[I 2026-08-12 19:57:14,099] Trial 2 finished with value: 0.9375691177708262 and parameters: {'n_estimators': 700, 'max_depth': 8, 'learning_rate': 0.23489105434827745, 'num_leaves': 112, 'sub

In [28]:
# Вывод лучших результатов
print("=== Лучшие гиперпараметры ===")
print(f"ROC-AUC: {study.best_value:.4f}")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Обновление пайплайна с лучшими параметрами
best_params_full = {f"classifier__{k}": v for k, v in study.best_params.items()}
best_params_full["classifier__scale_pos_weight"] = scale_pos_weight
best_params_full["classifier__random_state"] = 42
best_params_full["classifier__n_jobs"] = -1
best_params_full["classifier__verbose"] = -1

lgb_pipeline.set_params(**best_params_full)
print("\nПайплайн обновлён с лучшими параметрами.")

=== Лучшие гиперпараметры ===
ROC-AUC: 0.9453
  n_estimators: 800
  max_depth: 4
  learning_rate: 0.06303022099494215
  num_leaves: 208
  subsample: 0.7615109147561605
  colsample_bytree: 0.6870694990877506
  min_child_samples: 23
  reg_alpha: 0.34705504217989896
  reg_lambda: 4.1789632313657386e-08

Пайплайн обновлён с лучшими параметрами.
